# CLIPScope (original pipeline, ImageNet-1k as ID)

In [26]:
!pip -q install ftfy regex nltk open_clip_torch huggingface_hub datasets
!pip -q install git+https://github.com/openai/CLIP.git

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [27]:
import glob, shutil, subprocess
from pathlib import Path
import numpy as np
import torch
import clip
from PIL import Image
from sklearn import metrics
from torch.utils.data import Dataset, DataLoader, Subset

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE)

ROOT  = Path("/kaggle/working/CLIPScope")
CACHE = ROOT / ".cache"
DATA  = Path("/tmp/data")
for p in (ROOT, CACHE, DATA):
    p.mkdir(parents=True, exist_ok=True)

MODEL_NAME = "ViT-B/16"
L          = 10000   # images per dataset
K          = 5000    # nouns taken from each end
PERCENTILE = 50      # 5% of 1000 classes
SEED       = 0

model, preprocess = clip.load(MODEL_NAME, DEVICE)

device: cuda


In [28]:
import shutil

hits = list(Path("/kaggle/input").rglob("sorted_nouns.pt"))
assert hits, "Dataset not attached"
SRC = hits[0].parent
print("using:", SRC)

ROOT.mkdir(parents=True, exist_ok=True)
CACHE.mkdir(parents=True, exist_ok=True)

for name in ["imagenet_class_clean.npy", "nouns.pt", "sorted_nouns.pt", "results.pt"]:
    if (SRC / name).exists():
        shutil.copy(SRC / name, ROOT / name)

for p in (SRC / ".cache").glob("*.pt"):
    if not p.name.endswith(".part.pt"):
        shutil.copy(p, CACHE / p.name)

missing = []
for n in ["imagenet", "places", "textures", "inaturalist", "sun"]:
    f = CACHE / f"{n}_{MODEL_NAME.replace('/', '')}_{L}_{SEED}.pt"
    ok = f.exists()
    print(f"{f.name:40s} {'OK' if ok else 'MISSING'}")
    if not ok:
        missing.append(f.name)
assert not missing, f"Missing caches (these would trigger a dataset download): {missing}"

using: /kaggle/input/datasets/achinthacbandara/clipscope-cache
imagenet_ViT-B16_10000_0.pt              OK
places_ViT-B16_10000_0.pt                OK
textures_ViT-B16_10000_0.pt              OK
inaturalist_ViT-B16_10000_0.pt           OK
sun_ViT-B16_10000_0.pt                   OK


## 1. Generate `imagenet_class_clean.npy`
`open_clip` ships the cleaned ImageNet class names from OpenAI's CLIP prompt-engineering notebook. These may differ in a few entries from the author's file.

In [29]:
npy_path = ROOT / "imagenet_class_clean.npy"

if npy_path.exists():
    imagenet_cls = np.load(npy_path, allow_pickle=True)
    print("loaded cached:", npy_path)
else:
    from open_clip.zero_shot_metadata import IMAGENET_CLASSNAMES
    imagenet_cls = np.array(list(IMAGENET_CLASSNAMES))
    np.save(npy_path, imagenet_cls)
    print("generated from open_clip and saved:", npy_path)

assert len(imagenet_cls) == 1000
print(imagenet_cls[:5])
print(imagenet_cls[-5:])

loaded cached: /kaggle/working/CLIPScope/imagenet_class_clean.npy
['tench' 'goldfish' 'great white shark' 'tiger shark' 'hammerhead shark']
['earth star fungus' 'hen of the woods mushroom' 'bolete' 'corn cob'
 'toilet paper']


## 2. Noun mining

In [31]:
FORCE_REBUILD = True
nouns_path  = ROOT / "nouns.pt"
sorted_path = ROOT / "sorted_nouns.pt"
if FORCE_REBUILD:
    nouns_path.unlink(missing_ok=True)
    sorted_path.unlink(missing_ok=True)

def build_nouns():
    if nouns_path.exists():
        return torch.load(nouns_path, weights_only=False)
    import nltk
    from nltk.corpus import wordnet as wn
    nltk.download("wordnet", quiet=True)
    # Alg. 1 line 8: all nouns in WordNet, no further filtering
    s = {l.replace("_", " ") for syn in wn.all_synsets("n") for l in syn.lemma_names()}
    nouns = sorted(s)
    torch.save(nouns, nouns_path)
    return nouns

@torch.no_grad()
def mine_nouns(nouns, id_classes, k=K_PCT):
    id_f = encode_texts([f"a photo of a {c}" for c in id_classes]).to(DEVICE)
    scores = []
    for s in range(0, len(nouns), 1024):
        f = encode_texts([f"a photo of a {w}" for w in nouns[s:s+1024]]).to(DEVICE)
        sim = f @ id_f.T
        # 5th percentile of -sim == k-th largest sim; d = -score
        scores.append(sim.topk(k, dim=1).values[:, -1].cpu())
        print(f"  {min(s+1024, len(nouns))}/{len(nouns)}", end="\r")
    scores = torch.cat(scores).numpy().astype(np.float32)
    order = np.argsort(scores, kind="stable")   # ascending: farthest from ID first
    return [str(nouns[i]) for i in order]

if sorted_path.exists():
    sorted_nouns = torch.load(sorted_path, weights_only=False)
else:
    nouns = build_nouns()
    print("candidate nouns:", len(nouns))
    sorted_nouns = mine_nouns(nouns, imagenet_cls.tolist())
    torch.save(sorted_nouns, sorted_path)

print("Total nouns:", len(sorted_nouns))
print("Farthest :", sorted_nouns[:10])
print("Nearest  :", sorted_nouns[-10:])

candidate nouns: 119034
Total nouns: 119034
Farthest : ['prognostic', 'prognostication', 'prognosis', "yellow adder's tongue", 'Henri Emile Benoit Matisse', 'purple-staining Cortinarius', "adder's tongue", 'Herculaneum', 'Calypso bulbosa', 'purple-fringed orchis']
Nearest  : ['specific', 'original', 'near thing', 'image', 'phot', 'piece of work', 'looking', 'thing', 'device', 'item']


## 3. Datasets
* **ID:** ImageNet-1k validation (50k images) from Hugging Face
* **OOD:** iNaturalist, SUN, Places, Textures (DTD)

Only `L` random images are used from each, and their CLIP features are cached.

In [32]:
from huggingface_hub import login, snapshot_download
from kaggle_secrets import UserSecretsClient

def ensure_imagenet_val():
    files = sorted(glob.glob(str(DATA / "imagenet/data/val*.parquet")))
    if files:
        return files
    login(UserSecretsClient().get_secret("HF_TOKEN"))
    snapshot_download("ILSVRC/imagenet-1k", repo_type="dataset",
                      allow_patterns=["data/val*"], local_dir=DATA / "imagenet")
    return sorted(glob.glob(str(DATA / "imagenet/data/val*.parquet")))

class HFImages(Dataset):
    def __init__(self, files, transform=None):
        from datasets import load_dataset
        self.d = load_dataset("parquet", data_files=files, split="train")
        self.transform = transform
    def __len__(self): return len(self.d)
    def __getitem__(self, i):
        img = self.d[int(i)]["image"].convert("RGB")
        return (self.transform(img) if self.transform else img), 0

URLS = {
    "inaturalist": "http://pages.cs.wisc.edu/~huangrui/imagenet_ood_dataset/iNaturalist.tar.gz",
    "sun":         "http://pages.cs.wisc.edu/~huangrui/imagenet_ood_dataset/SUN.tar.gz",
    "places":      "http://pages.cs.wisc.edu/~huangrui/imagenet_ood_dataset/Places.tar.gz",
    "textures":    "https://www.robots.ox.ac.uk/~vgg/data/dtd/download/dtd-r1.0.1.tar.gz",
}

def ensure_ood(name):
    d = DATA / name
    if d.exists() and any(d.rglob("*.jpg")):
        return d
    d.mkdir(parents=True, exist_ok=True)
    tgz = DATA / f"{name}.tar.gz"
    subprocess.run(["wget", "-q", "-O", str(tgz), URLS[name]], check=True)
    subprocess.run(["tar", "-xzf", str(tgz), "-C", str(d)], check=True)
    tgz.unlink()
    return d

class FlatFolder(Dataset):
    def __init__(self, root, transform=None):
        exts = {".jpg", ".jpeg", ".png"}
        self.files = sorted(p for p in Path(root).rglob("*") if p.suffix.lower() in exts)
        self.transform = transform
    def __len__(self): return len(self.files)
    def __getitem__(self, i):
        img = Image.open(self.files[i]).convert("RGB")
        return (self.transform(img) if self.transform else img), 0

def get_dataset(name):
    if name == "imagenet":
        return HFImages(ensure_imagenet_val())
    return FlatFolder(ensure_ood(name))

def free_dataset(name):
    """Delete raw images from the scratch disk once features are cached."""
    shutil.rmtree(DATA / name, ignore_errors=True)

## 4. Image feature extraction

In [33]:
@torch.no_grad()
def image_features(name, n=L, seed=SEED, bs=128):
    cache = CACHE / f"{name}_{MODEL_NAME.replace('/', '')}_{n}_{seed}.pt"
    if cache.exists():
        return torch.load(cache, weights_only=False)

    part = cache.with_suffix(".part.pt")
    feats, done = [], 0
    if part.exists():
        d = torch.load(part, weights_only=False)
        feats, done = [d["f"]], d["n"]
        print(f"  resuming {name} from {done}")

    ds = get_dataset(name)
    ds.transform = preprocess
    idx = np.random.RandomState(seed).permutation(len(ds))[:min(n, len(ds))].tolist()
    dl = DataLoader(Subset(ds, idx[done:]), batch_size=bs, num_workers=4, pin_memory=True)

    for b, (x, _) in enumerate(dl):
        f = model.encode_image(x.to(DEVICE)).float()
        f /= f.norm(dim=-1, keepdim=True)
        feats.append(f.cpu())
        done += len(x)
        print(f"  [{name}] {done}/{len(idx)}", end="\r")
        if (b + 1) % 20 == 0:
            torch.save({"f": torch.cat(feats), "n": done}, part)

    feats = torch.cat(feats)
    torch.save(feats, cache)
    part.unlink(missing_ok=True)
    print()
    return feats

## 5. CLIPScope scoring

In [34]:
G_NAMES = ["p1", "p1/p0", "p2", "p2/p0", "p1*p2", "p1*p2/p0"]

def pick_labels(nouns, M, mode):
    if mode == "both":
        return nouns[:M] + nouns[-M:]
    elif mode == "farthest":
        return nouns[:M]
    elif mode == "closest":
        return nouns[-M:]
    else:
        raise ValueError(f"Unknown label mode: {mode}")

def build_text_features(id_classes, ood_labels):
    id_f   = encode_texts([f"a photo of a {c}" for c in id_classes])
    ood_f  = encode_texts([f"a photo of a {c}" for c in ood_labels])
    return torch.cat([id_f, ood_f], 0)

def make_order(kind, n_id, n_ood, rng):
    n = n_id + n_ood   
    if kind == "forward":
        return np.arange(n)
    elif kind == "reverse":
        return np.r_[np.arange(n_id, n), np.arange(n_id)]
    return rng.permutation(n)  # default random

@torch.no_grad()
def stream_scores(id_f, ood_f, text_f, n_id, rng, order="random"):
    x = torch.cat([id_f, ood_f]).to(DEVICE)
    y = np.r_[np.zeros(len(id_f)), np.ones(len(ood_f))]
    sim = 100 * (x @ text_f.to(DEVICE).T)

    probs = sim[:, :n_id].softmax(dim=-1)
    p1, pred = probs.max(dim=-1)
    p2 = sim.softmax(dim=-1)[:, :n_id].sum(dim=-1)

    p1 = p1.cpu().numpy()
    p2 = p2.cpu().numpy()
    pred = pred.cpu().numpy()

    count, total = np.ones(n_id), float(n_id)
    p0 = np.zeros(len(y))

    for i in make_order(order, len(id_f), len(ood_f), rng):
        c = pred[i]
        p0[i] = count[c] / total
        count[c] += 1
        total += 1

    scores = {
        "p1": p1,
        "p1/p0": p1 / p0,
        "p2": p2,
        "p2/p0": p2 / p0,
        "p1*p2": p1 * p2,
        "p1*p2/p0": p1 * p2 / p0
    }

    return scores, y

def evaluate(scores, labels):
    fpr, tpr, _ = metrics.roc_curve(labels, scores, pos_label=0)
    auroc = metrics.auc(fpr, tpr)
    valid = np.flatnonzero(tpr > 0.95)
    fpr95 = fpr[valid[0]] if len(valid) else np.nan
    return auroc, fpr95

def run_ood(ood_name, trials=5, M=K, label_mode="both", order="random"):
    id_classes = imagenet_cls.tolist()
    ood_labels = pick_labels(sorted_nouns, M, label_mode)
    text_f = build_text_features(id_classes, ood_labels)

    id_f = image_features("imagenet")
    ood_f = image_features(ood_name)

    n_trials = trials if order == "random" else 1
    res = {g: [] for g in G_NAMES}

    for t in range(n_trials):
        scores, y = stream_scores(
            id_f, ood_f, text_f, len(id_classes),
            np.random.RandomState(t), order
        )
        for g in G_NAMES:
            res[g].append(evaluate(scores[g], y))

    out = {}

    print(f"\n== {order} order | ID=ImageNet | OOD={ood_name} "
          f"| M={M}, labels={label_mode}, trials={n_trials} ==")

    for g in G_NAMES:
        a = np.array(res[g]) * 100
        out[g] = {
            "AUROC": a[:, 0].mean(),
            "AUROC_std": a[:, 0].std(),
            "FPR95": a[:, 1].mean(),
            "FPR95_std": a[:, 1].std()
        }
        print(
            f"g = {g:9s} "
            f"AUROC {out[g]['AUROC']:6.2f} +/- {out[g]['AUROC_std']:.2f}   "
            f"FPR95 {out[g]['FPR95']:6.2f} +/- {out[g]['FPR95_std']:.2f}"
        )

    return out

## 6. Experiments

In [35]:
results = {}
results[("places","random","both")] = run_ood("places", M=5000, label_mode="both", order="random")
torch.save(results, ROOT / "results.pt")
free_dataset("places")


== random order | ID=ImageNet | OOD=places | M=5000, labels=both, trials=5 ==
g = p1        AUROC  81.98 +/- 0.00   FPR95  59.56 +/- 0.00
g = p1/p0     AUROC  93.07 +/- 0.05   FPR95  28.25 +/- 0.20
g = p2        AUROC  86.88 +/- 0.00   FPR95  60.50 +/- 0.00
g = p2/p0     AUROC  93.28 +/- 0.03   FPR95  30.71 +/- 0.29
g = p1*p2     AUROC  87.55 +/- 0.00   FPR95  50.97 +/- 0.00
g = p1*p2/p0  AUROC  93.36 +/- 0.03   FPR95  30.00 +/- 0.20


In [36]:
results[("places","forward","both")] = run_ood("places", M=5000, label_mode="both", order="forward")
torch.save(results, ROOT / "results.pt")


== forward order | ID=ImageNet | OOD=places | M=5000, labels=both, trials=1 ==
g = p1        AUROC  81.98 +/- 0.00   FPR95  59.56 +/- 0.00
g = p1/p0     AUROC  91.20 +/- 0.00   FPR95  30.65 +/- 0.00
g = p2        AUROC  86.88 +/- 0.00   FPR95  60.50 +/- 0.00
g = p2/p0     AUROC  92.15 +/- 0.00   FPR95  33.85 +/- 0.00
g = p1*p2     AUROC  87.55 +/- 0.00   FPR95  50.97 +/- 0.00
g = p1*p2/p0  AUROC  92.16 +/- 0.00   FPR95  32.58 +/- 0.00


In [37]:
results[("places","reverse","both")] = run_ood("places", M=5000, label_mode="both", order="reverse")
torch.save(results, ROOT / "results.pt")


== reverse order | ID=ImageNet | OOD=places | M=5000, labels=both, trials=1 ==
g = p1        AUROC  81.98 +/- 0.00   FPR95  59.56 +/- 0.00
g = p1/p0     AUROC  93.15 +/- 0.00   FPR95  26.00 +/- 0.00
g = p2        AUROC  86.88 +/- 0.00   FPR95  60.50 +/- 0.00
g = p2/p0     AUROC  93.81 +/- 0.00   FPR95  26.90 +/- 0.00
g = p1*p2     AUROC  87.55 +/- 0.00   FPR95  50.97 +/- 0.00
g = p1*p2/p0  AUROC  94.12 +/- 0.00   FPR95  26.55 +/- 0.00


In [38]:
results[("places","random","closest")] = run_ood("places", M=5000, label_mode="closest", order="random")
torch.save(results, ROOT / "results.pt")


== random order | ID=ImageNet | OOD=places | M=5000, labels=closest, trials=5 ==
g = p1        AUROC  81.98 +/- 0.00   FPR95  59.56 +/- 0.00
g = p1/p0     AUROC  93.07 +/- 0.05   FPR95  28.25 +/- 0.20
g = p2        AUROC  86.61 +/- 0.00   FPR95  65.68 +/- 0.00
g = p2/p0     AUROC  93.03 +/- 0.04   FPR95  32.34 +/- 0.40
g = p1*p2     AUROC  87.22 +/- 0.00   FPR95  54.74 +/- 0.00
g = p1*p2/p0  AUROC  93.12 +/- 0.03   FPR95  31.13 +/- 0.23


In [39]:
results[("places","random","farthest")] = run_ood("places", M=5000, label_mode="farthest", order="random")
torch.save(results, ROOT / "results.pt")


== random order | ID=ImageNet | OOD=places | M=5000, labels=farthest, trials=5 ==
g = p1        AUROC  81.98 +/- 0.00   FPR95  59.56 +/- 0.00
g = p1/p0     AUROC  93.07 +/- 0.05   FPR95  28.25 +/- 0.20
g = p2        AUROC  83.71 +/- 0.00   FPR95  66.75 +/- 0.00
g = p2/p0     AUROC  91.55 +/- 0.03   FPR95  38.50 +/- 0.35
g = p1*p2     AUROC  85.74 +/- 0.00   FPR95  51.27 +/- 0.00
g = p1*p2/p0  AUROC  93.14 +/- 0.02   FPR95  32.29 +/- 0.22


In [40]:
results[("textures","random","both")] = run_ood("textures", M=5000, label_mode="both", order="random")
torch.save(results, ROOT / "results.pt")
free_dataset("textures")


== random order | ID=ImageNet | OOD=textures | M=5000, labels=both, trials=5 ==
g = p1        AUROC  79.54 +/- 0.00   FPR95  62.73 +/- 0.00
g = p1/p0     AUROC  93.13 +/- 0.07   FPR95  30.54 +/- 0.69
g = p2        AUROC  68.23 +/- 0.00   FPR95  85.50 +/- 0.00
g = p2/p0     AUROC  85.64 +/- 0.11   FPR95  60.77 +/- 0.29
g = p1*p2     AUROC  74.41 +/- 0.00   FPR95  74.47 +/- 0.00
g = p1*p2/p0  AUROC  87.21 +/- 0.08   FPR95  55.73 +/- 0.40


In [41]:
results[("textures","forward","both")] = run_ood("textures", M=5000, label_mode="both", order="forward")
torch.save(results, ROOT / "results.pt")


== forward order | ID=ImageNet | OOD=textures | M=5000, labels=both, trials=1 ==
g = p1        AUROC  79.54 +/- 0.00   FPR95  62.73 +/- 0.00
g = p1/p0     AUROC  89.90 +/- 0.00   FPR95  38.97 +/- 0.00
g = p2        AUROC  68.23 +/- 0.00   FPR95  85.50 +/- 0.00
g = p2/p0     AUROC  81.20 +/- 0.00   FPR95  67.55 +/- 0.00
g = p1*p2     AUROC  74.41 +/- 0.00   FPR95  74.47 +/- 0.00
g = p1*p2/p0  AUROC  83.85 +/- 0.00   FPR95  60.35 +/- 0.00


In [42]:
results[("textures","reverse","both")] = run_ood("textures", M=5000, label_mode="both", order="reverse")
torch.save(results, ROOT / "results.pt")


== reverse order | ID=ImageNet | OOD=textures | M=5000, labels=both, trials=1 ==
g = p1        AUROC  79.54 +/- 0.00   FPR95  62.73 +/- 0.00
g = p1/p0     AUROC  94.40 +/- 0.00   FPR95  22.52 +/- 0.00
g = p2        AUROC  68.23 +/- 0.00   FPR95  85.50 +/- 0.00
g = p2/p0     AUROC  89.51 +/- 0.00   FPR95  48.40 +/- 0.00
g = p1*p2     AUROC  74.41 +/- 0.00   FPR95  74.47 +/- 0.00
g = p1*p2/p0  AUROC  90.58 +/- 0.00   FPR95  46.79 +/- 0.00


In [43]:
results[("textures","random","closest")] = run_ood("textures", M=5000, label_mode="closest", order="random")
torch.save(results, ROOT / "results.pt")


== random order | ID=ImageNet | OOD=textures | M=5000, labels=closest, trials=5 ==
g = p1        AUROC  79.54 +/- 0.00   FPR95  62.73 +/- 0.00
g = p1/p0     AUROC  93.13 +/- 0.07   FPR95  30.54 +/- 0.69
g = p2        AUROC  72.01 +/- 0.00   FPR95  86.70 +/- 0.00
g = p2/p0     AUROC  86.98 +/- 0.10   FPR95  59.68 +/- 0.29
g = p1*p2     AUROC  76.72 +/- 0.00   FPR95  75.66 +/- 0.00
g = p1*p2/p0  AUROC  88.33 +/- 0.07   FPR95  55.60 +/- 0.58


In [44]:
results[("textures","random","farthest")] = run_ood("textures", M=5000, label_mode="farthest", order="random")
torch.save(results, ROOT / "results.pt")


== random order | ID=ImageNet | OOD=textures | M=5000, labels=farthest, trials=5 ==
g = p1        AUROC  79.54 +/- 0.00   FPR95  62.73 +/- 0.00
g = p1/p0     AUROC  93.13 +/- 0.07   FPR95  30.54 +/- 0.69
g = p2        AUROC  64.54 +/- 0.00   FPR95  87.93 +/- 0.00
g = p2/p0     AUROC  86.13 +/- 0.12   FPR95  64.99 +/- 0.50
g = p1*p2     AUROC  74.41 +/- 0.00   FPR95  74.80 +/- 0.00
g = p1*p2/p0  AUROC  89.39 +/- 0.08   FPR95  51.71 +/- 0.25


In [45]:
results[("inaturalist","random","both")] = run_ood("inaturalist", M=5000, label_mode="both", order="random")
torch.save(results, ROOT / "results.pt")
free_dataset("inaturalist")


== random order | ID=ImageNet | OOD=inaturalist | M=5000, labels=both, trials=5 ==
g = p1        AUROC  87.53 +/- 0.00   FPR95  49.30 +/- 0.00
g = p1/p0     AUROC  98.12 +/- 0.02   FPR95   7.88 +/- 0.15
g = p2        AUROC  99.37 +/- 0.00   FPR95   2.82 +/- 0.00
g = p2/p0     AUROC  99.49 +/- 0.00   FPR95   2.01 +/- 0.06
g = p1*p2     AUROC  99.34 +/- 0.00   FPR95   3.03 +/- 0.00
g = p1*p2/p0  AUROC  99.58 +/- 0.00   FPR95   1.72 +/- 0.03


In [46]:
results[("inaturalist","forward","both")] = run_ood("inaturalist", M=5000, label_mode="both", order="forward")
torch.save(results, ROOT / "results.pt")


== forward order | ID=ImageNet | OOD=inaturalist | M=5000, labels=both, trials=1 ==
g = p1        AUROC  87.53 +/- 0.00   FPR95  49.30 +/- 0.00
g = p1/p0     AUROC  97.68 +/- 0.00   FPR95   9.86 +/- 0.00
g = p2        AUROC  99.37 +/- 0.00   FPR95   2.82 +/- 0.00
g = p2/p0     AUROC  99.52 +/- 0.00   FPR95   2.17 +/- 0.00
g = p1*p2     AUROC  99.34 +/- 0.00   FPR95   3.03 +/- 0.00
g = p1*p2/p0  AUROC  99.57 +/- 0.00   FPR95   1.95 +/- 0.00


In [47]:
results[("inaturalist","reverse","both")] = run_ood("inaturalist", M=5000, label_mode="both", order="reverse")
torch.save(results, ROOT / "results.pt")


== reverse order | ID=ImageNet | OOD=inaturalist | M=5000, labels=both, trials=1 ==
g = p1        AUROC  87.53 +/- 0.00   FPR95  49.30 +/- 0.00
g = p1/p0     AUROC  98.10 +/- 0.00   FPR95   7.42 +/- 0.00
g = p2        AUROC  99.37 +/- 0.00   FPR95   2.82 +/- 0.00
g = p2/p0     AUROC  99.44 +/- 0.00   FPR95   2.21 +/- 0.00
g = p1*p2     AUROC  99.34 +/- 0.00   FPR95   3.03 +/- 0.00
g = p1*p2/p0  AUROC  99.57 +/- 0.00   FPR95   1.73 +/- 0.00


In [48]:
results[("inaturalist","random","closest")] = run_ood("inaturalist", M=5000, label_mode="closest", order="random")
torch.save(results, ROOT / "results.pt")


== random order | ID=ImageNet | OOD=inaturalist | M=5000, labels=closest, trials=5 ==
g = p1        AUROC  87.53 +/- 0.00   FPR95  49.30 +/- 0.00
g = p1/p0     AUROC  98.12 +/- 0.02   FPR95   7.88 +/- 0.15
g = p2        AUROC  92.61 +/- 0.00   FPR95  44.65 +/- 0.00
g = p2/p0     AUROC  97.94 +/- 0.01   FPR95   9.98 +/- 0.13
g = p1*p2     AUROC  93.25 +/- 0.00   FPR95  37.19 +/- 0.00
g = p1*p2/p0  AUROC  98.32 +/- 0.01   FPR95   8.31 +/- 0.11


In [49]:
results[("inaturalist","random","farthest")] = run_ood("inaturalist", M=5000, label_mode="farthest", order="random")
torch.save(results, ROOT / "results.pt")


== random order | ID=ImageNet | OOD=inaturalist | M=5000, labels=farthest, trials=5 ==
g = p1        AUROC  87.53 +/- 0.00   FPR95  49.30 +/- 0.00
g = p1/p0     AUROC  98.12 +/- 0.02   FPR95   7.88 +/- 0.15
g = p2        AUROC  99.65 +/- 0.00   FPR95   1.18 +/- 0.00
g = p2/p0     AUROC  99.65 +/- 0.00   FPR95   1.05 +/- 0.02
g = p1*p2     AUROC  99.69 +/- 0.00   FPR95   0.90 +/- 0.00
g = p1*p2/p0  AUROC  99.74 +/- 0.00   FPR95   0.78 +/- 0.03


In [50]:
results[("sun","random","both")] = run_ood("sun", M=5000, label_mode="both", order="random")
torch.save(results, ROOT / "results.pt")
free_dataset("sun")


== random order | ID=ImageNet | OOD=sun | M=5000, labels=both, trials=5 ==
g = p1        AUROC  83.99 +/- 0.00   FPR95  57.62 +/- 0.00
g = p1/p0     AUROC  96.69 +/- 0.06   FPR95  14.75 +/- 0.34
g = p2        AUROC  91.33 +/- 0.00   FPR95  45.36 +/- 0.00
g = p2/p0     AUROC  97.14 +/- 0.02   FPR95  14.11 +/- 0.19
g = p1*p2     AUROC  91.33 +/- 0.00   FPR95  38.17 +/- 0.00
g = p1*p2/p0  AUROC  97.00 +/- 0.02   FPR95  14.93 +/- 0.12


In [51]:
results[("sun","forward","both")] = run_ood("sun", M=5000, label_mode="both", order="forward")
torch.save(results, ROOT / "results.pt")


== forward order | ID=ImageNet | OOD=sun | M=5000, labels=both, trials=1 ==
g = p1        AUROC  83.99 +/- 0.00   FPR95  57.62 +/- 0.00
g = p1/p0     AUROC  95.70 +/- 0.00   FPR95  17.28 +/- 0.00
g = p2        AUROC  91.33 +/- 0.00   FPR95  45.36 +/- 0.00
g = p2/p0     AUROC  96.47 +/- 0.00   FPR95  17.05 +/- 0.00
g = p1*p2     AUROC  91.33 +/- 0.00   FPR95  38.17 +/- 0.00
g = p1*p2/p0  AUROC  96.13 +/- 0.00   FPR95  17.74 +/- 0.00


In [52]:
results[("sun","reverse","both")] = run_ood("sun", M=5000, label_mode="both", order="reverse")
torch.save(results, ROOT / "results.pt")


== reverse order | ID=ImageNet | OOD=sun | M=5000, labels=both, trials=1 ==
g = p1        AUROC  83.99 +/- 0.00   FPR95  57.62 +/- 0.00
g = p1/p0     AUROC  96.78 +/- 0.00   FPR95  13.26 +/- 0.00
g = p2        AUROC  91.33 +/- 0.00   FPR95  45.36 +/- 0.00
g = p2/p0     AUROC  97.51 +/- 0.00   FPR95  11.77 +/- 0.00
g = p1*p2     AUROC  91.33 +/- 0.00   FPR95  38.17 +/- 0.00
g = p1*p2/p0  AUROC  97.61 +/- 0.00   FPR95  11.93 +/- 0.00


In [53]:
results[("sun","random","closest")] = run_ood("sun", M=5000, label_mode="closest", order="random")
torch.save(results, ROOT / "results.pt")


== random order | ID=ImageNet | OOD=sun | M=5000, labels=closest, trials=5 ==
g = p1        AUROC  83.99 +/- 0.00   FPR95  57.62 +/- 0.00
g = p1/p0     AUROC  96.69 +/- 0.06   FPR95  14.75 +/- 0.34
g = p2        AUROC  90.86 +/- 0.00   FPR95  52.81 +/- 0.00
g = p2/p0     AUROC  97.07 +/- 0.02   FPR95  15.69 +/- 0.20
g = p1*p2     AUROC  90.80 +/- 0.00   FPR95  44.73 +/- 0.00
g = p1*p2/p0  AUROC  96.91 +/- 0.02   FPR95  16.17 +/- 0.18


In [54]:
results[("sun","random","farthest")] = run_ood("sun", M=5000, label_mode="farthest", order="random")
torch.save(results, ROOT / "results.pt")


== random order | ID=ImageNet | OOD=sun | M=5000, labels=farthest, trials=5 ==
g = p1        AUROC  83.99 +/- 0.00   FPR95  57.62 +/- 0.00
g = p1/p0     AUROC  96.69 +/- 0.06   FPR95  14.75 +/- 0.34
g = p2        AUROC  89.49 +/- 0.00   FPR95  51.14 +/- 0.00
g = p2/p0     AUROC  96.31 +/- 0.02   FPR95  20.18 +/- 0.38
g = p1*p2     AUROC  90.34 +/- 0.00   FPR95  36.55 +/- 0.00
g = p1*p2/p0  AUROC  97.02 +/- 0.02   FPR95  15.31 +/- 0.28


In [55]:
import pandas as pd

rows = []

for (dataset, order, label_mode), scores in results.items():

    df = pd.DataFrame(scores).T
    df.index.name = "Scoring"

    df = df.reset_index()

    df.insert(0, "Order", order)
    df.insert(0, "Label Mode", label_mode)
    df.insert(0, "Dataset", dataset)

    rows.append(df)

table = pd.concat(rows, ignore_index=True)
pd.set_option("display.max_rows", None)
display(table.round(2))

,Dataset,Label Mode,Order,Scoring,AUROC,AUROC_std,FPR95,FPR95_std
0,places,both,random,p1,81.98,0.00,59.56,0.00
1,places,both,random,p1/p0,93.07,0.05,28.25,0.20
2,places,both,random,p2,86.88,0.00,60.50,0.00
3,places,both,random,p2/p0,93.28,0.03,30.71,0.29
4,places,both,random,p1*p2,87.55,0.00,50.97,0.00
5,places,both,random,p1*p2/p0,93.36,0.03,30.00,0.20
6,places,both,forward,p1,81.98,0.00,59.56,0.00
7,places,both,forward,p1/p0,91.20,0.00,30.65,0.00
8,places,both,forward,p2,86.88,0.00,60.50,0.00
9,places,both,forward,p2/p0,92.15,0.00,33.85,0.00


In [ ]:
# Bundle the small cache files to download (or use Save Version)
shutil.make_archive("/kaggle/working/clipscope_cache", "zip", ROOT)
!du -sh /kaggle/working/clipscope_cache.zip